In [1]:
import numpy as np
import scipy.signal as signal
import scanpy as sc
from tqdm import tqdm
import pandas as pd
import anndata as ad

In [3]:
def detect_peaks(spectrum, mz_axis, prominence=0.01):
    """Detect peaks with prominence and return m/z and intensity."""
    peaks, properties = signal.find_peaks(spectrum, prominence=prominence)
    return mz_axis[peaks], spectrum[peaks]

def ppm_diff(ref_mz, target_mz):
    return (target_mz - ref_mz) / ref_mz * 1e6

def find_top_peaks_global(adata, prominence=0.01, n_peaks=100, ppm_tolerance=10):
    all_peaks = []

    # Use var_names (index) as the m/z axis
    mz_axis = adata.var_names.astype(float).values

    for i in tqdm(range(adata.shape[0]), desc="Detecting peaks"):
        spectrum = adata.X[i].toarray().flatten()
        mz_peaks, intensities = detect_peaks(spectrum, mz_axis, prominence)
        all_peaks.extend(list(zip(mz_peaks, intensities)))

    # Convert to structured array and sort by intensity
    all_peaks = np.array(all_peaks, dtype=[("mz", float), ("intensity", float)])
    sorted_peaks = np.sort(all_peaks, order="intensity")[::-1]

    # Select top N, removing duplicates within ±ppm_tolerance
    selected_peaks = []
    for peak in sorted_peaks:
        if len(selected_peaks) >= n_peaks:
            break
        if all(abs(ppm_diff(p["mz"], peak["mz"])) > ppm_tolerance for p in selected_peaks):
            selected_peaks.append(peak)

    return np.sort(np.array([p["mz"] for p in selected_peaks]))

def extract_and_aggregate_peaks(adata, selected_mz, ppm_tolerance=10):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    # Store results
    aggregated_intensities = []
    aggregated_mz_names = []

    for target_mz in selected_mz:
        # Find all indices within the ppm tolerance
        ppm_diff_array = np.abs((mz_axis - target_mz) / target_mz * 1e6)
        within_tolerance = np.where(ppm_diff_array <= ppm_tolerance)[0]

        if len(within_tolerance) == 0:
            # No peaks found within tolerance, fill with zeros
            summed_intensity = np.zeros((adata.shape[0],))
            avg_mz = target_mz  # fallback to target_mz
        else:
            # Sum intensities across matched peaks
            summed = X[:, within_tolerance].sum(axis=1)
            summed_intensity = np.asarray(summed).flatten()
            avg_mz = mz_axis[within_tolerance].mean()

        aggregated_intensities.append(summed_intensity)
        aggregated_mz_names.append(f"{avg_mz:.4f}")

    # Stack the results to form a new data matrix
    new_X = np.vstack(aggregated_intensities).T  # shape: (n_obs, n_selected_mz)

    # Create new AnnData object
    new_adata = ad.AnnData(
        X=new_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=aggregated_mz_names)
    )

    return new_adata


In [4]:
# File paths
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_1000_10ppm.h5ad"

# Parameters
prominence = 0.01
top_n = 1000
ppm_tolerance = 10

# Load and process
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

selected_mz = find_top_peaks_global(adata, prominence=prominence, n_peaks=top_n, ppm_tolerance=ppm_tolerance)
adata_top_peaks = extract_and_aggregate_peaks(adata, selected_mz, ppm_tolerance=ppm_tolerance)

# Save the reduced AnnData
adata_top_peaks.write(output_file)
print(f"Reduced AnnData with top {top_n} peaks saved to: {output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad


Detecting peaks: 100%|██████████| 16605/16605 [01:50<00:00, 150.56it/s]


Reduced AnnData with top 1000 peaks saved to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_1000_10ppm.h5ad


In [5]:
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_1000_10ppm.h5ad"
adata = sc.read_h5ad(input_file)

In [7]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [6]:
adata.var

""
96.9261
112.8980
114.8924
114.8971
136.0608
...
1074.0223
1079.0364
1495.1523
1947.5227


In [8]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [9]:
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'

In [10]:
import plotly.graph_objects as go
import numpy as np

def plot_aggregated_peaks(adata, title="Summed Intensities per Averaged m/z"):
    # Get m/z and summed intensities
    mz_axis = adata.var_names.astype(float).values

    # Sum across all observations (pixels) to get total intensity per m/z
    if hasattr(adata.X, "toarray"):  # sparse
        summed_intensity = np.asarray(adata.X.sum(axis=0)).ravel()
    else:  # dense
        summed_intensity = np.sum(adata.X, axis=0)

    # Sort by intensity (descending) and keep top 1000
    top_indices = np.argsort(summed_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = summed_intensity[top_indices]

    # Sort m/z for prettier plotting
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Plot using Plotly
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='lines+markers',
        marker=dict(size=4, color='darkblue'),
        line=dict(width=1),
        hovertemplate='Avg m/z: %{x:.4f}<br>Summed Intensity: %{y:.2f}',
        name='Summed Intensity'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Averaged m/z",
        yaxis_title="Summed Intensity",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()

In [13]:
plot_aggregated_peaks(adata_top_peaks)


In [14]:
import plotly.graph_objects as go
import numpy as np

def interactive_stem_plot_summed_intensity(adata, top_n=1000):
    # Get m/z axis
    mz_axis = adata.var_names.astype(float).values

    # Sum intensities across all observations
    if hasattr(adata.X, "toarray"):  # sparse
        summed_intensity = np.asarray(adata.X.sum(axis=0)).ravel()
    else:
        summed_intensity = np.sum(adata.X, axis=0)

    # Get top N by intensity
    top_indices = np.argsort(summed_intensity)[::-1][:top_n]
    mz_top = mz_axis[top_indices]
    intensity_top = summed_intensity[top_indices]

    # Sort by m/z for prettier plot
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Create the stem plot
    fig = go.Figure()

    # Add vertical stems
    for x, y in zip(mz_top, intensity_top):
        fig.add_trace(go.Scatter(
            x=[x, x],
            y=[0, y],
            mode='lines',
            line=dict(color='darkblue', width=1),
            hoverinfo='skip',
            showlegend=False
        ))

    # Add markers on top
    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='markers',
        marker=dict(size=4, color='darkblue'),
        hovertemplate='Avg m/z: %{x:.4f}<br>Summed Intensity: %{y:.2f}',
        name='Summed Intensity'
    ))

    fig.update_layout(
        title=f"Interactive Stem Plot of Top {top_n} Summed Intensities",
        xaxis_title="Averaged m/z",
        yaxis_title="Summed Intensity",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()


In [15]:
interactive_stem_plot_summed_intensity(adata_top_peaks, top_n=1000)

In [19]:
import plotly.graph_objects as go
import numpy as np

def interactive_stem_spectrum_pixel(adata, pixel_index, mz_min=None, mz_max=None):
    # Extract m/z axis and spectrum
    mz_axis = adata.var_names.astype(float).values
    spectrum = adata.X[pixel_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[pixel_index]

    # Filter by mz range if provided
    if mz_min is not None and mz_max is not None:
        mask = (mz_axis >= mz_min) & (mz_axis <= mz_max)
        mz_axis = mz_axis[mask]
        spectrum = spectrum[mask]

    # Create the stem lines (from 0 to intensity)
    fig = go.Figure()

    for mz, intensity in zip(mz_axis, spectrum):
        fig.add_trace(go.Scatter(
            x=[mz, mz],
            y=[0, intensity],
            mode="lines",
            line=dict(color="blue", width=1),
            hoverinfo="skip"
        ))

    # Add the tips of the stems
    fig.add_trace(go.Scatter(
        x=mz_axis,
        y=spectrum,
        mode='markers',
        marker=dict(color='red', size=5),
        hovertemplate='m/z: %{x:.4f}<br>Intensity: %{y:.2f}',
        name='Peaks'
    ))

    fig.update_layout(
        title=f"Interactive Stem Plot: Spectrum at Pixel {pixel_index}",
        xaxis_title="m/z",
        yaxis_title="Intensity",
        template="plotly_white",
        height=400,
        hovermode="closest"
    )

    fig.show()

In [20]:
interactive_stem_spectrum_pixel(adata, pixel_index=4500, mz_min=100, mz_max=2000)